# Оценка Aniemore/wavlm-emotion-russian-resd на RESD test

Загружает модель с HuggingFace, прогоняет на тестовой части `Aniemore/resd`,  
выводит Accuracy, Weighted Accuracy, F1 macro, classification report.

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'datasets', 'torchaudio', 'scikit-learn',
], check=True)
print('Done.')

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, classification_report,
)
from tqdm.auto import tqdm

MODEL_NAME = 'Aniemore/wavlm-emotion-russian-resd'
SR_TARGET  = 16_000
BATCH_SIZE = 8

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
print(f'Loading model: {MODEL_NAME}')
processor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model     = AutoModelForAudioClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

# метки модели
id2label = model.config.id2label
print(f'Labels: {id2label}')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
print('Loading Aniemore/resd test split...')
ds_test = load_dataset('Aniemore/resd', split='test')
print(f'Test samples: {len(ds_test)}')
print(f'Features: {ds_test.features}')

In [ ]:
# маппинг меток датасета → id модели
label2id_model = {v.lower(): k for k, v in id2label.items()}

all_preds, all_labels = [], []

for i in tqdm(range(0, len(ds_test), BATCH_SIZE), desc='Evaluating'):
    batch = ds_test.select(range(i, min(i + BATCH_SIZE, len(ds_test))))

    wavs = []
    for ex in batch:
        wav = np.array(ex['speech']['array'], dtype=np.float32)
        sr  = ex['speech']['sampling_rate']
        if sr != SR_TARGET:
            import torchaudio.functional as F_audio
            wav = F_audio.resample(
                torch.tensor(wav), orig_freq=sr, new_freq=SR_TARGET).numpy()
        wavs.append(wav)

    inputs = processor(wavs, sampling_rate=SR_TARGET, return_tensors='pt',
                       padding=True).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
    preds = logits.argmax(-1).cpu().tolist()

    labels = [label2id_model[ex['emotion'].lower()] for ex in batch]

    all_preds.extend(preds)
    all_labels.extend(labels)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
print(f'Evaluated {len(all_preds)} samples.')

In [ ]:
label_names = [id2label[i] for i in sorted(id2label)]

print('=== Results: Aniemore/wavlm-emotion-russian-resd on RESD test ===')
print(f'Accuracy          : {accuracy_score(all_labels, all_preds):.4f}')
print(f'Weighted Accuracy  : {balanced_accuracy_score(all_labels, all_preds):.4f}')
print(f'F1 Macro          : {f1_score(all_labels, all_preds, average="macro",    zero_division=0):.4f}')
print(f'F1 Weighted       : {f1_score(all_labels, all_preds, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=label_names, zero_division=0))